# Optimized Classification Model with Feature Importance Analysis

This notebook trains an optimized classification model (Random Forest, tuned with `GridSearchCV`) on the **Breast Cancer Wisconsin (Diagnostic)** dataset, evaluates it, and analyzes which features matter most.

**Dataset source:** loaded directly from a public URL (see README.md for the link) — no manual upload needed, works out of the box in Google Colab.


## 1. Install & Import Libraries

In [ ]:
!pip install -q scikit-learn pandas matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

sns.set_style("whitegrid")
RANDOM_STATE = 42


## 2. Load Dataset

The dataset is fetched directly from a public URL, so this cell works as-is in Google Colab.
Replace `DATA_URL` with your own dataset link if you want to use a different dataset (see README.md).


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/plotly/datasets/master/breast-cancer-wisconsin-data.csv"

df = pd.read_csv(DATA_URL)
df.head()


In [ ]:
print("Shape:", df.shape)
df.info()


## 3. Clean & Prepare Data

In [ ]:
# Drop non-informative / empty columns commonly found in this dataset
drop_cols = [c for c in ["id", "Unnamed: 32"] if c in df.columns]
df = df.drop(columns=drop_cols)

# Encode target: diagnosis -> M (malignant) = 1, B (benign) = 0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

df = df.dropna()

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("Features:", X.shape[1])
print("Class balance:\n", y.value_counts(normalize=True))


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x=y)
plt.title("Target Class Distribution (0=Benign, 1=Malignant)")
plt.xlabel("Diagnosis")
plt.show()


In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(X.corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()


## 5. Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)


## 6. Model Optimization (Hyperparameter Tuning)

We use `GridSearchCV` with stratified k-fold cross-validation to find the best Random Forest configuration.


In [ ]:
param_grid = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC: %.4f" % grid_search.best_score_)

best_model = grid_search.best_estimator_


## 7. Evaluate the Optimized Model

In [ ]:
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test ROC-AUC:  {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Benign", "Malignant"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Optimized Random Forest")
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


## 8. Feature Importance Analysis

Two complementary views:
1. **Built-in Random Forest importance** (mean decrease in impurity)
2. **Permutation importance** (more reliable — measures the actual drop in performance when a feature is shuffled)


In [ ]:
importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10,8))
sns.barplot(x=importances.values[:15], y=importances.index[:15], palette="viridis")
plt.title("Top 15 Features - Random Forest Importance (MDI)")
plt.xlabel("Importance")
plt.show()

importances.head(15)


In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    best_model, X_test_scaled, y_test,
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)

perm_importances = pd.Series(perm_result.importances_mean, index=X.columns)
perm_importances = perm_importances.sort_values(ascending=False)

plt.figure(figsize=(10,8))
sns.barplot(x=perm_importances.values[:15], y=perm_importances.index[:15], palette="magma")
plt.title("Top 15 Features - Permutation Importance")
plt.xlabel("Mean Decrease in ROC-AUC")
plt.show()

perm_importances.head(15)


## 9. Save the Optimized Model

In [ ]:
import joblib

joblib.dump(best_model, "optimized_classification_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")

print("Model and scaler saved.")


## 10. Summary

- Trained and tuned a Random Forest classifier via `GridSearchCV` (5-fold stratified CV, optimized for ROC-AUC).
- Achieved strong test accuracy and ROC-AUC (see printed metrics above).
- Identified the most predictive features using both impurity-based and permutation-based importance — useful for understanding what drives the model's decisions and for potential feature selection in future iterations.

**Next steps:** try other model families (XGBoost, Logistic Regression) for comparison, or apply SHAP for per-prediction explainability.
